# Workshop: Fine-Tune NVIDIA VISTA3D on Databricks AI Runtime with Genie Code

**Before you start:** clone this notebook into your own workspace folder (**File → Clone**). You will work in your copy: Genie Code adds the code cells under each step.

## What you will build
You will fine-tune **NV-Segment-CTMR (VISTA3D)**, NVIDIA's 3D medical segmentation foundation model from the MONAI ecosystem, to segment **7 heart structures** in congenital heart disease CT scans. Then you will:
* compare the model **before and after** fine-tuning, with Dice scores and overlay images
* register the fine-tuned model in **Unity Catalog**, in your own schema `<catalog>.ws_<your_name>`

Training runs on a **serverless H100 GPU** provisioned by Databricks **AI Runtime**. This notebook itself needs **no GPU**.

## How this workshop works: you + Genie Code + skills
You don't write the code yourself. Every step gives you a prompt for **Genie Code** (Agent mode). Genie Code uses two **skills**, packaged instructions plus tested scripts:

| Skill | What it gives Genie Code |
| --- | --- |
| `nv-segment-finetune` | A pinned model registry, project templates, a CPU validator, the GPU-side training launcher, before/after evaluation, plotting and Unity Catalog registration helpers |
| `nv-reason-finetune` | The same for the NV-Reason vision-language models (used in the instructor demo, Appendix A) |

A skill is a folder with a `SKILL.md`. Genie Code reads it when your request matches its description, then follows its workflow and runs its scripts. It doesn't improvise training code. **Each step explains which component Genie Code uses and why**, so you understand everything that runs. Read Genie Code's plan before approving each cell.

## The model and the data
**NV-Segment-CTMR** covers 345 CT and MRI anatomy classes. We use the **softmax** fine-tuning workflow: it turns the model into a fixed 8-channel segmenter (background + 7 structures). Every structure starts from a class the model already knows:

| Dataset label | Structure | VISTA3D class ID |
| --- | --- | --- |
| 1 | Left ventricle | 151 |
| 2 | Right ventricle | 152 |
| 3 | Left atrium | 149 |
| 4 | Right atrium | 153 |
| 5 | Left ventricle myocardium | 154 |
| 6 | Aorta | 6 |
| 7 | Pulmonary artery | 171 |

**ImageCHD** is a public dataset of 110 3D CT volumes with voxel-level masks, stored in a Unity Catalog Volume as `ct_XXXX_image.nii.gz` / `ct_XXXX_label.nii.gz` pairs.

## Key concepts
| Term | Meaning |
| --- | --- |
| **NIfTI** (`.nii.gz`) | Standard 3D medical image format. CT intensities are in **Hounsfield units (HU)**: air −1000, water 0, contrast-filled blood ≈ 200–400 |
| **Label mask** | A 3D volume where each voxel holds a class number (0 = background) |
| **VISTA3D class ID** | The model's own number for a structure (e.g. 151 = left ventricle). The **label mapping** links your dataset labels to these IDs |
| **Softmax vs standard workflow** | Softmax: a fixed set of non-overlapping classes, one output channel each. Standard: keeps VISTA3D's promptable head and can learn brand-new classes |
| **Dice score** | Overlap between prediction and ground truth, from 0 (none) to 1 (perfect) |
| **Epoch / patch** | One pass over the training cases / the 3D crop (e.g. 96³ voxels) the model sees at a time |
| **AI Runtime** | Serverless GPU compute for training jobs, submitted with the `air` CLI from a `train.yaml` |
| **MLflow run** | The record of one training job: parameters, metrics, files ("artifacts") and the model |

## Agenda (≈ 2 hours)
| Step | What you do | Time |
| --- | --- | --- |
| 0 | Set up: libraries, variables, your Unity Catalog schema | 10 min |
| 1 | Explore the data | 10 min |
| 2 | Generate the training project with the skill | 10 min |
| 3 | Understand the project files | 15 min |
| 4 | Validate locally and size the run for the session | 10 min |
| 5 | Submit to AI Runtime | 5 min (+ 20–30 min training) |
| 6 | Monitor the run | while it trains |
| 7 | Compare before vs after | 15 min |
| 8 | Register the model in Unity Catalog | 10 min |
| 9 | Wrap-up, clean-up, next steps | 5 min |

If your job hasn't finished by Step 7, use the instructor's **reference run** (`REFERENCE_RUN_ID`, shared at the start) for Steps 7–8, and come back to your own run later.

## Architecture
```
┌───────────────────────────────────────────────────────────────────────┐
│ This notebook (CPU)          you + Genie Code + skills               │
│   explore data · generate project · validate · submit                │
│   before/after plots · register in Unity Catalog                     │
└──────────────────────────────────┬────────────────────────────────────┘
             air run (snapshot of│the project folder)
                                 ▼
┌───────────────────────────────────────────────────────────────────────┐
│ AI Runtime (serverless GPU_1xH100)   train_segment_finetune.py       │
│   1. install pinned dependencies (monai 1.4.0 …)                     │
│   2. stage data · clone NV-Segment-CTMR @ pinned commit · download   │
│      pretrained weights @ pinned revision                            │
│   3. train: python -m monai.bundle run (upstream softmax config)     │
│   4. evaluate before (pretrained) vs after (fine-tuned)              │
│   5. log metrics, checkpoint, evaluation and an MLflow model         │
└──────────────────────────────────┬────────────────────────────────────┘
                                 ▼
             MLflow experiment ──► Unity Catalog: <catalog>.ws_<you>.<model>
```

## Prerequisites

**Instructor (once, before the workshop)**
* Upload ImageCHD to a Unity Catalog Volume, e.g. `/Volumes/pixels_workshop_catalog/image_chd/default/imagechd/extracted/ImageCHD_dataset`
* Install both skills workspace-wide: copy `nv-segment-finetune/` and `nv-reason-finetune/` to `/Workspace/.assistant/skills/`
* Grant the participants' group `USE CATALOG` and `CREATE SCHEMA` on the workshop catalog, and `READ VOLUME` on the dataset volume
* Make sure AI Runtime is enabled with enough `GPU_1xH100` capacity (or a GPU pool) for the class
* Run the workshop once, share its MLflow run ID as `REFERENCE_RUN_ID`, and give the participants' group **CAN READ** on that run's experiment, so they can plot and register it

**Participant**
* Workspace access with **Genie Code in Agent mode**
* This notebook cloned to your folder, attached to **serverless** (CPU) compute
* The values the instructor shares: `CATALOG`, `DATASET_PATH`, `REFERENCE_RUN_ID`

**Licensing and use.** NV-Segment models are released under NVIDIA's model license (see the Hugging Face model card). Everything here is for engineering and research use, not clinical use.

## Step 0: Set Up — libraries, variables, your Unity Catalog schema

### Goal
Install the notebook-side libraries once, define your variables in one cell, and create your personal schema.

### Ask Genie Code
> **0a.** Add a cell that runs `%pip install databricks-air nibabel matplotlib`, followed by a cell with `%restart_python`.
>
> **0b.** Add one configuration cell that sets: `USERNAME` from the notebook context; `USER_ID` = the part of my username before `@`, lowercased, with non-alphanumerics replaced by `_`; `CATALOG = "<catalog from the instructor>"`; `SCHEMA = f"ws_{USER_ID}"`; `MODEL_ID = "nv-segment-ctmr"`; `SKILL_DIR = "/Workspace/.assistant/skills/nv-segment-finetune"`; `PROJECT_DIR = f"/Workspace/Users/{USERNAME}/air-segment-workshop"`; `DATASET_PATH = "<path from the instructor>"`; `EXPERIMENT_NAME = f"/Users/{USERNAME}/Segment-Finetune-Workshop"`; `GPU_TYPE = "GPU_1xH100"`; `REFERENCE_RUN_ID = "<from the instructor>"`. Then `import sys; sys.path.insert(0, f"{SKILL_DIR}/scripts")` and print every value.
>
> **0c.** Create my Unity Catalog schema with `ensure_schema(CATALOG, SCHEMA)` from the nv-segment-finetune skill's `register_model` script, then run `SHOW SCHEMAS IN <CATALOG> LIKE '<SCHEMA>'` to confirm it exists.

### What Genie Code does — and why
* **`%pip install` + `%restart_python`** — `databricks-air` is the AI Runtime CLI you use in Step 5; `nibabel` reads NIfTI files and `matplotlib` draws the images. Installing everything first means Python restarts **once**, so no variables are lost later.
* **One configuration cell** — every later step reads these variables. If Python restarts, re-run this single cell.
* **`SKILL_DIR` on `sys.path`** — makes the skill's helper scripts (`visualize.py`, `register_model.py`) importable, so they run in this notebook with your credentials.
* **`ensure_schema`** — the same as `CREATE SCHEMA IF NOT EXISTS <catalog>.ws_<you>`, done through the Databricks SDK. If you lack permission, it returns a clear message instead of a stack trace. Unity Catalog organises everything as `catalog.schema.object`. Your own schema is your sandbox: your registered model goes there, isolated from other participants'.

### What you should see
* The printed variables, with `SCHEMA` like `ws_jane_doe`
* Your schema in **Catalog Explorer** under the workshop catalog
* If schema creation fails with a permission error, ask the instructor for `USE CATALOG` / `CREATE SCHEMA`

## Step 1: Explore the Data

### Goal
Confirm you can read the dataset, and see what the model will learn before any training.

### Ask Genie Code
> Count the `ct_*_image.nii.gz` and `ct_*_label.nii.gz` files in `DATASET_PATH` and list the first five. Then use `visualize.show_case` from the nv-segment-finetune skill to display `ct_1001` with its label overlay, passing the 7 class names (1 left ventricle … 7 pulmonary artery). Repeat with `view="coronal"`.

### What Genie Code does — and why
* **`glob` over the Volume** — UC Volumes appear as normal paths under `/Volumes/...`, so ordinary Python file tools work, governed by Unity Catalog permissions.
* **`visualize.show_case`** — loads the CT and its mask with `nibabel`, reorients both to a standard orientation, and picks the slice with the most labelled voxels. It shows the CT in a soft-tissue window (−200 to 600 HU) and overlays each structure in a fixed, colour-blind-checked colour. It also prints the volume shape, HU range and voxels per class.

### What you should see
* **110 images and 110 labels**
* A CT slice through the heart, with the chambers, myocardium, aorta and pulmonary artery coloured
* Voxel counts per class: the myocardium and pulmonary artery are small structures, so they are usually the hardest to learn

## Step 2: Generate the Training Project with the Skill

### Goal
Create the AI Runtime project, four files in `PROJECT_DIR`, without writing training code.

### Ask Genie Code
> Use the nv-segment-finetune skill to create an AI Runtime project for `MODEL_ID` in `PROJECT_DIR`, with `DATASET_PATH`, `EXPERIMENT_NAME` and `GPU_TYPE`, and check the dataset. Show me the JSON report: each file's action, any warnings, and whether validation passed. Do not submit the job.

### What Genie Code does — and why
* It runs the skill's **`scripts/scaffold_project.py`** rather than writing files itself. The scaffold copies reviewed templates from the skill's `assets/` folder and fills them from the **model registry**. The registry pins the exact upstream code commit, the pretrained weights revision, and the library versions, so every participant trains the same, reproducible setup.
* It writes **four files**, explained in Step 3: `train.yaml`, `finetune_config.yml`, `train_segment_finetune.py` and `segment_pyfunc.py`.
* It then runs the **validator** (`scripts/validate_project.py`) on CPU and returns a JSON report. Existing files are never overwritten unless you ask for `--force`.

### What you should see
* `"ok": true`, with four files `created`
* `"workflow": "softmax"`, and the `next_step` hint: `air run --file train.yaml`
* The four files in your workspace folder

## Step 3: Understand the Project Files

### Goal
Know what each file controls, so you can change the run safely.

### Ask Genie Code
> Print `train.yaml` and `finetune_config.yml` from `PROJECT_DIR`, and explain each section in two lines. Then summarise what `train_segment_finetune.py` does step by step, and what `segment_pyfunc.py` is for, using the skill's `references/monai-bundle-contract.md`.

### What Genie Code does — and why
It reads the generated files and the skill's reference document, which records the upstream MONAI bundle's documented contract. The four files have separate jobs:

| File | Role | Key points |
| --- | --- | --- |
| `train.yaml` | **Infrastructure**, read by the `air` CLI | `compute` = which GPU; `environment.version: '5'` = base image with PyTorch 2.9 + CUDA (quoted string); `dependencies` = pinned extras such as `monai==1.4.0`. **Never** `torch`: it ships with the environment. `command` = what runs on the GPU (`cd $CODE_SOURCE_PATH`, the folder where your snapshot is unpacked). `code_source` = snapshot this folder |
| `finetune_config.yml` | **What to train** | `model` = model ID, workflow and pins; `dataset` = files, label mapping, `check_labels`; `training` = epochs, learning rate, patch size; `evaluation.max_cases` = how many validation cases get a before/after comparison; `mlflow` = where results go |
| `train_segment_finetune.py` | **The launcher** that runs on the GPU | Stages data into the MONAI layout, splits off validation cases, and clones the upstream bundle at its pinned commit. It downloads the pretrained weights and runs the **upstream** entry point `python -m monai.bundle run`; it doesn't reimplement training. Afterwards it evaluates before vs after and logs everything to MLflow |
| `segment_pyfunc.py` | **The MLflow model wrapper** | Packages checkpoint + label mapping + bundle configs into a model that takes an `image_path` and returns a mask path. This is what you register in Unity Catalog in Step 8 |

**Settings merge in priority order:** command-line flags > `finetune_config.yml` > model defaults > launcher defaults. So `--epochs 1` at the end of the `command` overrides the file.

**About the label mapping:** the mapping order defines the output channels (channel 0 = background). Never add "padding" entries such as `[8, 0]`; each entry becomes a channel. Stray label values in your files are set to background automatically (`check_labels: true`).

### What you should see
Genie Code's explanation, matching the table above. Ask follow-up questions now; it's cheaper than after submitting.

## Step 4: Validate Locally and Size the Run for the Session

### Goal
Catch mistakes on CPU, and make the job short enough to finish during the workshop.

### Ask Genie Code
> In `PROJECT_DIR/finetune_config.yml`, set `dataset.max_cases: 30` and `training.epochs: 10`, keeping everything else. Then run the nv-segment-finetune validator with `--check-dataset` and show every check. Finally run `train_segment_finetune.py --config finetune_config.yml --print-config` from `PROJECT_DIR` and confirm the resolved epochs, learning rate, label mapping and evaluation cases.

### What Genie Code does — and why
* **Workshop sizing** — 30 cases × 10 epochs trains in roughly 20–30 minutes; the full recipe (110 cases × 50 epochs) takes 2–3 hours. Fewer epochs means a lower final Dice; the before/after comparison still shows the effect clearly.
* **`scripts/validate_project.py`** runs on CPU, without MONAI or a GPU. It checks:
  * that the files exist, the YAML parses, and `environment.version` is a quoted string
  * that `torch` is not pinned and the model's required pins are present
  * that the GPU count matches the GPU type
  * that the label mapping is valid and every class ID exists in the pretrained model, which the softmax workflow requires
  * that the dataset has matching image/label pairs
* **`--print-config`** makes the launcher resolve its final settings, merging the file, defaults and flags, then exit without downloading anything.

### What you should see
* Every check `✓` and `"ok": true`; fix any `✗` before continuing
* The resolved config with `epochs: 10`, `max_cases: 30`, `eval_cases: 3` and the 7-entry label mapping

## Step 5: Submit to AI Runtime

### Goal
Launch the training job on a serverless H100.

### Ask Genie Code
> Export my Databricks credentials for subprocesses: set `DATABRICKS_HOST` and `DATABRICKS_TOKEN` from the notebook context and print the host and a masked token. Then run `air run --file train.yaml` with `subprocess.run` from `PROJECT_DIR`, passing `os.environ.copy()`, without `--watch`, and print the output including the run URL.

### What Genie Code does — and why
* **Credential export** — `air` runs as a separate process, which does **not** inherit the notebook's login. The notebook context's API URL and short-lived token are passed as environment variables; Genie Code never prints the full token.
* **`air run --file train.yaml`** — the AI Runtime CLI:
  1. snapshots `PROJECT_DIR` and uploads it
  2. provisions the GPU from `compute`
  3. installs `dependencies`
  4. runs `command`, which starts `train_segment_finetune.py` on the GPU node
* **No `--watch`** — it would stream logs for the whole run and time out the cell. You monitor in Step 6 instead.

The Databricks CLI (v1.18+) offers the same command as `databricks air run --file train.yaml` for terminal use.

### What you should see
A **Job Run URL**. Open it in a new tab. Provisioning takes 2–10 minutes.

## Step 6: Monitor the Run

### Goal
Follow the job and find its MLflow run.

### Ask Genie Code
> Look up the MLflow experiment `EXPERIMENT_NAME` and list the 5 most recent runs with run ID, status, start time, `best_val_mean_dice` and the `segment.model_id` tag. Store the newest run's ID as `RUN_ID`.

### What Genie Code does — and why
* **`mlflow.search_runs`** reads the experiment the launcher logs to (the `mlflow.experiment` value in `finetune_config.yml`). The launcher creates one MLflow run per job.
* **What the job does, visible in the Job Run logs:**
  1. **Stage data** — link the cases into the MONAI layout, set stray label values to background, and write `datalist.json` with a train/validation split
  2. **Stage bundle** — clone NV-Segment-CTMR at the pinned commit and download the pretrained weights
  3. **Train** — run the upstream softmax workflow, which validates every epoch (epoch 0 scores the starting weights)
  4. **Before/after evaluation** — on 3 validation cases, see Step 7
  5. **Log the MLflow model** — the `segment_pyfunc.py` wrapper
* **MLflow content:** parameters, `val_mean_dice` / `train_loss` per epoch, `best_val_mean_dice`, `eval_before_mean_dice` / `eval_after_mean_dice`, and artifacts: `checkpoints/`, `configs/`, `evaluation/`, `model/`, `run_summary.json`, `train.log`.

### What you should see
* Status `RUNNING`, then `FINISHED`; open the run in the MLflow UI to watch `val_mean_dice` rise
* If your job is still running when the class moves on, set `RUN_ID = REFERENCE_RUN_ID` for Steps 7–8

## Step 7: Compare Before vs After Fine-Tuning

### Goal
See and measure what fine-tuning changed.

### Ask Genie Code
> Use the nv-segment-finetune skill's `visualize` helpers. Download the evaluation for `RUN_ID`, show the per-class Dice table, plot the Dice chart, and show `plot_before_after` for each evaluated case in the axial and coronal views. Then summarise which structures improved most and which are still weak.

### What Genie Code does — and why
* **`visualize.load_evaluation(RUN_ID)`** downloads the run's `evaluation/` artifacts, which the GPU job produced, so everything here runs on CPU:
  * `dice.json` — Dice per class, per case, before and after
  * `slices_<case>.npz` — small 2D slices of CT, ground truth, before and after masks
* **How "before" and "after" were produced** — on the same held-out validation cases, both through the upstream MONAI inference configs:
  * **Before** = the pretrained model, zero-shot: it was asked to find the 7 structures by class ID (`configs/inference.json` with label prompts)
  * **After** = your fine-tuned softmax model (`configs/inference_softmax.json`)
* **`dice_table` / `plot_dice`** — mean Dice per structure, before vs after; the table is the exact numbers behind the chart.
* **`plot_before_after`** — four panels: CT | ground truth | before | after, at the slice with the most labelled anatomy.

### What you should see
* **After** Dice higher than **before** for most structures; small, thin structures (myocardium, pulmonary artery) usually gain the most
* In the images, the fine-tuned masks follow the ground-truth boundaries more closely
* With the short workshop run, absolute Dice is lower than the full 50-epoch recipe. Compare with the reference run.

Three validation cases and training-time validation are an engineering check, not a clinical evaluation.

## Step 8: Register the Model in Unity Catalog

### Goal
Publish your fine-tuned model as a governed, versioned asset in your schema.

### Ask Genie Code
> With the nv-segment-finetune skill, run `register_run(RUN_ID, CATALOG, schema=SCHEMA, dry_run=True)` and show the resolved model name and URI. If it looks right, run it again without `dry_run`, then show the registered model's versions, the `workshop` alias, the tags, and the input/output signature.

### What Genie Code does — and why
* **`register_model.register_run`** runs in this notebook, so it uses your credentials:
  1. checks the run finished successfully
  2. finds the MLflow model the launcher logged (run tag `segment.model_uri`)
  3. makes sure `CATALOG.SCHEMA` exists
  4. registers the model as `CATALOG.ws_<you>.nv_segment_ctmr_finetuned`
* **Version + alias** — every registration creates a new version. The alias `workshop` points at the one you just made, so code can load `models:/<name>@workshop` without hard-coding a number.
* **Tags and description** — the version records the upstream commit, the weights revision, the workflow, the source run and the before/after Dice. Anyone can trace how it was built.
* **Signature** — Unity Catalog requires one. Input: `image_path` (a NIfTI path). Outputs: `label_path` (the predicted mask) and `class_voxels` (voxel counts per structure). Running the model needs a GPU (a job or a GPU serving endpoint); registering it does not.

### What you should see
* Version **1** (or the next number, if you register again) of `CATALOG.ws_<you>.nv_segment_ctmr_finetuned`, with alias `workshop`
* In **Catalog Explorer → your schema → Models**: the version, its tags, and lineage back to the MLflow run

## Step 9: Wrap-up, Clean-up and Next Steps

### What you built
A reproducible AI Runtime project, a fine-tuned VISTA3D model with a measured before/after improvement, and a governed model version in your own Unity Catalog schema. Genie Code did the typing; the skill kept every step on the upstream, pinned, validated path.

### Clean-up (ask Genie Code)
* Cancel any run still going: **Cancel** in the Jobs UI, or `databricks air cancel <run>` from a terminal
* Optional: delete your workshop schema when you no longer need it: `DROP SCHEMA <catalog>.ws_<you> CASCADE`

### Common errors and fixes
| Error | Cause | Fix |
| --- | --- | --- |
| Validator `✗` on a check | Edited file broke a rule | Read the check's detail; ask Genie Code to fix it with the skill |
| `softmax needs existing bundle class IDs` | A class ID the pretrained model doesn't know | Use the `standard` workflow, or pick IDs from the skill's `assets/label_ids/` |
| `air run` authentication error | Credentials not exported | Re-run the export from Step 5 |
| `INTERNAL_ERROR` at provisioning | GPU capacity | Retry, or ask the instructor about `GPU_1xA100` or a pool |
| CUDA out of memory | Patch or cache too large | `training.patch_size: [96,96,96]` or smaller; lower `cache_rate` |
| Old code runs on the GPU | Files changed after submitting | Re-submit; each run uploads a fresh snapshot |
| `Cannot create <catalog>.ws_<you>` | Missing permission | Ask for `USE CATALOG` / `CREATE SCHEMA` |
| Variables gone | Python restarted | Re-run the Step 0b configuration cell |

### Next steps
* **Full-quality run** — restore `max_cases: null`, `epochs: 50` and re-submit (2–3 h)
* **Batch inference** — load `models:/<catalog>.ws_<you>.nv_segment_ctmr_finetuned@workshop` in a GPU job and segment new scans from a Volume
* **Your own data** — change `dataset.src_dir`, the file pattern and `label_mapping` in `finetune_config.yml`; the validator tells you whether the classes fit the softmax workflow
* **Another model** — NV-Segment-CT works the same way (Appendix B); the NV-Reason models are shown in Appendix A

## Appendix A — Instructor Demo: Fine-Tuning NV-Reason-CXR and NV-Reason-CT

*Demonstrated by the instructor. These models need credentialed data and, by default, 8 GPUs per run.*

### What these models are
**Vision-language models** that read a chest X-ray (NV-Reason-CXR-3B, on Qwen2.5-VL) or a 3D chest/abdominal CT (NV-Reason-CT, on Qwen3.5 + a 3D image encoder) and write a reasoning trace plus findings. Unlike VISTA3D, they are trained with Hugging Face **TRL**:
* **SFT** (supervised fine-tuning) — imitate reference answers
* **GRPO** — sample several answers and learn from a reward on the verified findings

| | NV-Reason-CXR-3B | NV-Reason-CT |
| --- | --- | --- |
| Input | 2D X-ray (JPG/PNG) | 3D CT NIfTI (HU) |
| Example data | MIMIC-CXR-JPG / CheXpert (credentialed) | CT-RATE (gated on Hugging Face) |
| Default GPUs | `GPU_8xH100` | `GPU_8xH100` (1 GPU only with `freeze_llm`) |
| License | NVIDIA OneWay **Noncommercial** | OpenMDW-1.1 |

### Demo flow (same pattern as Steps 2–7, with the `nv-reason-finetune` skill)
1. **Generate** — *"Use the nv-reason-finetune skill to create an SFT project for nv-reason-ct in `PROJECT_DIR` with `MANIFEST`, `IMAGE_DIR` and `EXPERIMENT_NAME` on GPU_8xH100, and validate it. Do not submit."*
   * **What happens:** the scaffold writes `train.yaml` with model-specific pins (`transformers==5.6.2`, `trl==1.2.0`, …) and a DeepSpeed install line, plus `finetune_config.yml` and `train_reason_finetune.py`. The validator checks every manifest row: SFT rows must end with the assistant answer, GRPO rows need a `solution`. It also checks that every referenced image exists.
2. **Explain** — the launcher runs the **upstream** `accelerate launch train/vlm_sft_train.py` with the repo's own config, plus AI Runtime overrides:
   * pinned weights downloaded first
   * MLflow instead of W&B
   * DeepSpeed ZeRO-2 on 8 GPUs
   * for CXR GRPO, vLLM off (it would downgrade PyTorch)
3. **Smoke run** — set `training.max_steps: 8` and `dataset.max_cases: 32`, then submit with `air run`.
4. **Before vs after** — the last `evaluation.max_cases` manifest rows are held out of training. After training, the job answers them with the base and the fine-tuned model:
   * NV-Reason-CT: the upstream `inference.py`
   * NV-Reason-CXR: the README's generation code

   Ask Genie Code: *"Use the nv-reason-finetune skill's `show_eval` helpers to show the findings-F1 table for `RUN_ID` and the first case with its image, reference, before and after answers."* Findings F1 compares the predicted `<answer>` findings with the verified `solution`; read the texts too.

Unity Catalog registration for these models is not part of this workshop. The fine-tuned weights and processor are logged under the run's `model/` artifacts.

## Appendix B — Using NV-Segment-CT Instead of NV-Segment-CTMR

NV-Segment-CT is the CT-only VISTA3D (124 classes). Its label dictionary has **no heart chambers**, so the softmax workflow can't start the ImageCHD classes from known IDs. Use the **standard** workflow: the aorta (ID 6) starts from its pretrained class, and the chambers are learned as new classes.

* Set `MODEL_ID = "nv-segment-ct"` in Step 0b, and use a new `PROJECT_DIR`
* Step 2 prompt: *"… create an AI Runtime project for nv-segment-ct with the standard workflow …"*. The validator warns that the chamber IDs are new to this model; that is expected.
* Step 7: "before" uses NVIDIA's formal evaluation config (`configs/evaluate.json`) with the pretrained weights. Classes the pretrained model has never seen have no meaningful "before" mask (or none at all, if that evaluation fails, which is reported as a warning). Compare "after" with the ground truth. This path has not been run on AI Runtime yet.